In [1]:
%pip install -U transformers accelerate "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 114.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.8 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.21.0
    Uninstalling transformers-5.21.0:
      Successfully uninstalled transformers-5.21.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bitsandbytes]


In [2]:
import sys
import torch
import bitsandbytes as bnb
import transformers
from transformers.utils import is_bitsandbytes_available

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers detects BitsAndBytes:", is_bitsandbytes_available())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
Transformers: 5.22.1
BitsAndBytes: 0.50.2
CUDA available: True
Transformers detects BitsAndBytes: True


In [3]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

In [4]:
import zipfile
import os

zip_path = "/content/legal-advisor-final-1.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

print(os.listdir("/content/legal-advisor-final-1"))

['README.md', 'chat_template.jinja', 'adapter_config.json', 'training_args.bin', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


In [5]:
BASE_MODEL = "Qwen/Qwen3-4B"
ADAPTER_PATH = "/content/legal-advisor-final-1"

In [6]:
import bitsandbytes as bnb


# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

# Quantize base model to 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [7]:
# Load your trained LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

RAG PIPELINE

In [34]:
%pip install -U langgraph langchain-core langchain-community langchain-huggingface sentence-transformers faiss-cpu rank-bm25 transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.1/577.1 kB 16.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 93.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 44.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 128.5 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.0
    Uninstalling langchain-core-1.7.0:
      Successfully uninstalled langchain-core-1.7.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0
  Attempting uninstall: langgraph
    Found existing install

In [35]:
import json
from pathlib import Path

JSON_PATH = Path("constitution_of_india.json")

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {JSON_PATH.resolve()}"
    )

with JSON_PATH.open("r", encoding="utf-8") as f:
    constitution_data = json.load(f)

# Support either a list or a dictionary containing records
if isinstance(constitution_data, dict):
    for key in ("articles", "data", "records"):
        if isinstance(constitution_data.get(key), list):
            constitution_data = constitution_data[key]
            break

if not isinstance(constitution_data, list):
    raise ValueError("Expected a list of Constitution records.")

required_fields = {"article", "title", "description"}

valid_records = [
    item for item in constitution_data
    if isinstance(item, dict)
    and required_fields.issubset(item.keys())
]

if not valid_records:
    raise ValueError(
        "No valid records found. Check the JSON field names."
    )

print("Total records:", len(valid_records))
print("Sample record:")
print(json.dumps(valid_records[0], indent=2, ensure_ascii=False))

Total records: 465
Sample record:
{
  "article": 0,
  "title": "Preamble",
  "description": "WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens: JUSTICE, social, economic and political; LIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity; and to promote among them all FRATERNITY assuring the dignity of the individual and the unity and integrity of the Nation; IN OUR CONSTITUENT ASSEMBLY this twenty-sixth day of November, 1949, do HEREBY ADOPT, ENACT AND GIVE TO OURSELVES THIS CONSTITUTION"
}


In [36]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

documents = []

for item in valid_records:
    article_no = item["article"]
    title = item["title"]
    description = item["description"]

    content = (
        f"Article: {article_no}\n"
        f"Title: {title}\n"
        f"Description:\n{description}"
    )

    documents.append(
        Document(
            page_content=content,
            metadata={
                "article": article_no,
                "title": title,
                "source": "constitution_of_india.json"
            }
        )
    )

print("Documents created:", len(documents))

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

/tmp/ipykernel_535/3863774157.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Documents created: 465


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [37]:
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

print("FAISS index created successfully.")
print("Indexed documents:", vector_store.index.ntotal)

FAISS index created successfully.
Indexed documents: 465


In [38]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

print("BM25 index created successfully.")

BM25 index created successfully.


In [41]:
def hybrid_retrieve(query, k=5, dense_k=10, sparse_k=10, rrf_k=60):
    # Dense retrieval using FAISS
    dense_results = vector_store.similarity_search(
        query, k=dense_k
    )

    # Sparse retrieval using the existing BM25 index
    bm25_retriever.k = sparse_k
    sparse_results = bm25_retriever.invoke(query)

    # Reciprocal Rank Fusion (RRF)
    scores = {}
    doc_lookup = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            key = (
                str(doc.metadata["article"]),
                doc.metadata["title"]
            )

            scores[key] = scores.get(key, 0) + 1 / (rrf_k + rank)
            doc_lookup[key] = doc

    ranked_keys = sorted(
        scores,
        key=lambda key: scores[key],
        reverse=True
    )

    return [
        {
            "document": doc_lookup[key],
            "rrf_score": scores[key]
        }
        for key in ranked_keys[:k]
    ]

In [43]:
query = "Which constitutional article guarantees equality before law?"

results = hybrid_retrieve(query, k=5)

for rank, result in enumerate(results, start=1):
    doc = result["document"]

    print(f"\nRank: {rank}")
    print("Article:", doc.metadata["article"])
    print("Title:", doc.metadata["title"])
    print("RRF score:", round(result["rrf_score"], 5))


Rank: 1
Article: 14
Title: Equality before law
RRF score: 0.03279

Rank: 2
Article: 16
Title: Equality of opportunity in matters of public employment
RRF score: 0.02964

Rank: 3
Article: 13
Title: Laws inconsistent with or in derogation of the fundamental rights
RRF score: 0.01613

Rank: 4
Article: 292
Title: Borrowing by the Government of India
RRF score: 0.01613

Rank: 5
Article: 305
Title: Saving of existing laws and laws providing for State monopolies
RRF score: 0.01587


In [44]:
test_cases = [
    {
        "query": "Which article guarantees equality before law?",
        "expected": 14
    },
    {
        "query": "Which article prohibits discrimination on grounds of religion, race, caste, sex or place of birth?",
        "expected": 15
    },
    {
        "query": "Which article protects life and personal liberty?",
        "expected": 21
    }
]

for test in test_cases:
    results = hybrid_retrieve(test["query"], k=5)

    retrieved_articles = [
        result["document"].metadata["article"]
        for result in results
    ]

    print("\nQuery:", test["query"])
    print("Expected article:", test["expected"])
    print("Retrieved articles:", retrieved_articles)
    print("Correct in top 5:", test["expected"] in retrieved_articles)


Query: Which article guarantees equality before law?
Expected article: 14
Retrieved articles: [14, 16, 305, 292, 293]
Correct in top 5: True

Query: Which article prohibits discrimination on grounds of religion, race, caste, sex or place of birth?
Expected article: 15
Retrieved articles: [15, 23, 325, 29, 16]
Correct in top 5: True

Query: Which article protects life and personal liberty?
Expected article: 21
Retrieved articles: [21, 27, '48A', '300A', 43]
Correct in top 5: True


In [45]:
from peft import PeftModel

# Check whether the model is a PEFT model
adapter_attached = isinstance(model, PeftModel)

print("LoRA adapter attached:", adapter_attached)

if adapter_attached:
    print("Active adapters:", model.active_adapters)

    # Check loaded adapter configuration
    print("Adapter names:", list(model.peft_config.keys()))

    for name, config in model.peft_config.items():
        print(f"\nAdapter: {name}")
        print("LoRA rank (r):", config.r)
        print("LoRA alpha:", config.lora_alpha)
        print("Target modules:", config.target_modules)
else:
    print(
        "The model variable is not a PeftModel. "
        "Check whether the adapter was loaded separately or merged into the base model."
    )

LoRA adapter attached: True
Active adapters: ['default']
Adapter names: ['default']

Adapter: default
LoRA rank (r): 16
LoRA alpha: 32
Target modules: {'up_proj', 'o_proj', 'down_proj', 'k_proj', 'q_proj', 'v_proj', 'gate_proj'}


In [46]:
# Inspect LoRA layers and their weight tensors
lora_layers = []

for name, module in model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        if "default" in module.lora_A and "default" in module.lora_B:
            lora_layers.append(name)

print("LoRA layers found:", len(lora_layers))
print("Sample layers:", lora_layers[:10])

assert len(lora_layers) > 0, "No LoRA weight layers found!"
print("PASS: LoRA layers and adapter weights are present.")

LoRA layers found: 252
Sample layers: ['base_model.model.model.layers.0.self_attn.q_proj', 'base_model.model.model.layers.0.self_attn.k_proj', 'base_model.model.model.layers.0.self_attn.v_proj', 'base_model.model.model.layers.0.self_attn.o_proj', 'base_model.model.model.layers.0.mlp.gate_proj', 'base_model.model.model.layers.0.mlp.up_proj', 'base_model.model.model.layers.0.mlp.down_proj', 'base_model.model.model.layers.1.self_attn.q_proj', 'base_model.model.model.layers.1.self_attn.k_proj', 'base_model.model.model.layers.1.self_attn.v_proj']
PASS: LoRA layers and adapter weights are present.


In [47]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END


class LegalRAGState(TypedDict):
    question: str
    search_query: str
    documents: List[Dict[str, Any]]
    relevant: bool
    retry_count: int
    answer: str

In [48]:
def retrieve_node(state: LegalRAGState):
    query = state["search_query"]

    results = hybrid_retrieve(query, k=5)

    retrieved_docs = []

    for item in results:
        doc = item["document"]

        retrieved_docs.append({
            "article": doc.metadata["article"],
            "title": doc.metadata["title"],
            "text": doc.page_content,
            "score": item["rrf_score"]
        })

    print(f"Retrieved {len(retrieved_docs)} documents for: {query}")

    return {"documents": retrieved_docs}

In [49]:
import re

STOPWORDS = {
    "what", "which", "who", "when", "where", "why", "how",
    "does", "do", "did", "is", "are", "was", "were", "the",
    "a", "an", "of", "in", "on", "to", "for", "and", "or",
    "under", "constitution", "article", "indian"
}


def tokenize(text):
    words = re.findall(r"[a-zA-Z0-9]+", text.lower())
    return {word for word in words if word not in STOPWORDS and len(word) > 2}


def evaluate_node(state: LegalRAGState):
    question_terms = tokenize(state["question"])

    # Include title and article text in the relevance check
    relevant = False

    for doc in state["documents"]:
        evidence_terms = tokenize(doc["title"] + " " + doc["text"])
        overlap = question_terms & evidence_terms

        if len(overlap) >= 2:
            relevant = True
            break

    print("Evidence relevance:", relevant)

    return {"relevant": relevant}

In [50]:
def rewrite_query_node(state: LegalRAGState):
    question = state["question"]

    # Simple fallback rewrite; can later be replaced with Qwen-generated rewrites
    rewritten = f"Indian Constitution legal provision concerning: {question}"

    print("Rewritten query:", rewritten)

    return {
        "search_query": rewritten,
        "retry_count": state["retry_count"] + 1
    }


def route_after_evaluation(state: LegalRAGState):
    if state["relevant"]:
        return "generate"

    if state["retry_count"] < 1:
        return "rewrite"

    return "generate"

In [67]:
import torch

def generate_node(state: LegalRAGState):
    question = state["question"]
    docs = state["documents"]

    if not state["relevant"] or not docs:
        return {
            "answer": (
                "I could not find sufficient relevant evidence "
                "in the retrieved Constitution text."
            )
        }

    context = "\n\n".join(
        f"Article {doc['article']} — {doc['title']}\n{doc['text']}"
        for doc in docs
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a legal information assistant for the Constitution "
                "of India. Answer only from the provided evidence. "
                "Give the article number, title, and a concise explanation. "
                "Do not narrate your process, repeat the question, or add "
                "commentary about how you found the answer. "
                "Answer in 1-3 complete sentences. "
                "Include the complete relevant constitutional provision. "
                "Never stop in the middle of a sentence. "
                "Do not output internal reasoning or <think> tags."
                "Stop after the final answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"Evidence:\n{context}\n\n"
                f"Question: {question}\n"
                "Give the final answer in 2-3 sentences."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    input_device = model.get_input_embeddings().weight.device
    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05
        )

    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Remove a complete think block if present
    if "</think>" in answer:
        answer = answer.split("</think>", 1)[1].strip()

    return {"answer": answer}

In [68]:
workflow = StateGraph(LegalRAGState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("evaluate", evaluate_node)
workflow.add_node("rewrite", rewrite_query_node)
workflow.add_node("generate", generate_node)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "evaluate")

workflow.add_conditional_edges(
    "evaluate",
    route_after_evaluation,
    {
        "generate": "generate",
        "rewrite": "rewrite"
    }
)

workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("generate", END)

legal_rag_app = workflow.compile()

print("Agentic RAG graph compiled successfully.")

Agentic RAG graph compiled successfully.


In [70]:
question = "Which article prohibits discrimination on grounds of religion, race, caste, sex or place of birth?"

result = legal_rag_app.invoke({
    "question": question,
    "search_query": question,
    "documents": [],
    "relevant": False,
    "retry_count": 0,
    "answer": ""
})

print(result["answer"])

Retrieved 5 documents for: Which article prohibits discrimination on grounds of religion, race, caste, sex or place of birth?
Evidence relevance: True
<think>
Okay, let's see. The user is asking which article prohibits discrimination based on religion, race, caste, sex, or place of birth. I need to check the provided evidence.

Looking through the articles, Article 15 has a description that mentions prohibition of discrimination on those grounds. The title is "Prohibition of discrimination on grounds of religion, race, caste, sex or place of birth". The first paragraph under Article 15 says the state can't discriminate on those grounds. Also, the description includes clauses (1) and (2) which align with the question.


In [59]:
print("Chat template available:", bool(tokenizer.chat_template))
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", repr(tokenizer.pad_token))

if tokenizer.chat_template:
    print("\nChat template preview:")
    print(tokenizer.chat_template[:1500])

Chat template available: True
EOS token: '<|im_end|>'
EOS token ID: 151645
PAD token: '<|endoftext|>'

Chat template preview:
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_to